# Noise-model-v2 sampler

Draw a rendered rotor-noise clip from either of the two **winning noise-model-v2
fits**, flying a rotor-speed trajectory **sampled from the fitted trajectory
model of the same rig** — so the clip is a plausible flight of that aircraft,
not a hand-drawn speed curve.

| rig | noise fit(s) | renderer |
|---|---|---|
| `dregon` | round-5 **uncalibrated** single-regime fit `round5/fits/dregon_room2_floor__flight_profile.json` | `render.render_noise` |
| `michaels` | round-3 per-regime pair `round3/fits/michaels_fly125_{standby,cruise}__flight.json` | `render.render_noise_regimes` (smoothstep on the slowest rotor, 45 → 65 rev/s) |

The DREGON rig is the **uncalibrated** one on purpose: the +3.75 dB comb pin of
the calibrated variant turned out to be unnecessary. `comb_offset_db` (default
`0.0`) is there if you want to put a level offset back by hand; it is applied to
a deep copy, never to the loaded fit.

Trajectories come from `data_processing.trajectory_model` — the `rps-traj-fits`
bundle the `fitted_traj` training streams read
(`conf/online_mix/traj_fitted_5050.yaml`), with `measurement_noise=False`
because the audio is rendered *from* these labels. Each rig flies its own fitted
trajectory by default; `traj_rig="posterior"` instead draws a **fresh drone from
the global rig hyperprior** (§ 4).

**The rate contract.** `render_noise(fit, rps_rev_s, sr=16000, ...)` wants the
carrier as `(4, T)` on its **own output grid** — one rotor-speed sample per
audio sample at 16 kHz — and interpolates that up to 64 kHz itself. The
trajectory fits are stated on a 100 Hz grid and hold nothing above 20 Hz, so
`sample_trajectory` simulates the state space at 100 Hz and interpolates to
16 kHz. A rendered Frame therefore carries `rps` (decimated to 100 Hz, what the
plots draw) and `rps_render` (the exact 16 kHz carrier the renderer consumed).

Logic lives in `noise_v2_sampler.py`; the cells below are thin.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src", ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np

from plots import dwym

import noise_v2_sampler as NS

plt.rcParams.update({"figure.dpi": 110})

## 0 · Provenance

What is loaded, from where, and at which commit.

In [ ]:
for name in NS.RIGS:
    rig = NS.load_rig(name)
    print(f"{name}: checkout {rig.repo_sha}, trajectory rig {rig.traj_rig!r}")
    for regime, path in rig.paths.items():
        prov = rig.provenance[regime]
        print(f"   [{regime}] {path}")
        print(f"            schema {prov['schema']}  fit written at git {prov['git'][:12]}")
print()
print("trajectory bundle:", NS.TRAJ_FITS, "->", NS.traj_rig_names())

## 1 · DREGON

The round-5 uncalibrated single-regime fit, flown on the fitted DREGON
trajectory.

In [ ]:
dregon = NS.load_rig("dregon")
NS.describe(dregon)

One 20 s airborne trajectory, seed 0. `describe_draw` says which drone was
drawn and how its speeds sit against the span the noise fit was identified on.

In [ ]:
traj_d = NS.sample_trajectory(dregon, seed=0, duration_s=20.0)
NS.describe_draw(traj_d)

Render it — one microphone, uncalibrated comb, absolute fitted units.

`NS.show` is `dwym`'s `audio` + `rps` route (spectrogram over the rotor-speed
track, plus the player) with one difference: the spectrogram's colour range is
clipped to the top 45 dB instead of autoscaled over the full ~145 dB of the
log-magnitude data, which is what makes the comb visible at all. The plain
one-liner `dwym(clip_d)` is in the next cell for comparison.

In [ ]:
clip_d = NS.render(dregon, traj_d, seed=0, n_mics=1)
NS.show(clip_d)

In [ ]:
dwym(clip_d)

The fit's own forward model against what the renderer actually put on the wire:
the expected periodogram `M` versus the realised one, mic 0, first 4 s.

DREGON's comb sits far under its floor — that is the campaign's own finding for
this rig, not a rendering artefact — so both curves are floor-shaped and the
lines show as small ripples on top.

In [ ]:
NS.expected_vs_realised(dregon, clip_d);

## 2 · Michael's

The round-3 per-regime candidate. `render_noise_regimes` renders the standby and
the cruise fit on the whole track with independent seeds and sums them with
`sqrt(1-w)` / `sqrt(w)`, so the composed **power** interpolates the two regimes'
levels across the 45–65 rev/s ramp band.

In [ ]:
michaels = NS.load_rig("michaels")
NS.describe(michaels)

In [ ]:
traj_m = NS.sample_trajectory(michaels, seed=0, duration_s=20.0)
NS.describe_draw(traj_m)
clip_m = NS.render(michaels, traj_m, seed=0, n_mics=1)
NS.show(clip_m)

In [ ]:
NS.expected_vs_realised(michaels, clip_m);

### 2.1 · A whole flight, standby → cruise

`full_flight=True` wraps the airborne process in `flight.wrap_airborne`: ground,
spin-up, warm-up idle, take-off, airborne, landing, spin-down, ground. That is
what drives the carrier through the 45–65 rev/s band, so the standby fit
actually contributes and the regime blend is exercised. `frac_in_blend` is the
share of samples strictly inside the ramp band; `cruise_weight` is
`[min, max, mean]` of the cruise weight.

In [ ]:
traj_full = NS.sample_trajectory(michaels, seed=1, duration_s=30.0, full_flight=True)
clip_full = NS.render(michaels, traj_full, seed=1, n_mics=1)

meta = dict(clip_full["meta"].items())
print("cruise weight [min, max, mean]:", [round(v, 3) for v in meta["cruise_weight"]])
print("fraction of samples inside the 45-65 rev/s blend:", round(meta["frac_in_blend"], 4))
NS.show(clip_full)

## 3 · Both rigs side by side

The same trajectory seed on both rigs — each flying **its own** fitted
trajectory model — rendered with the same render seed, stacked on a shared time
axis.

In [ ]:
SEED, DUR = 7, 12.0

both = {}
for rig in (dregon, michaels):
    traj = NS.sample_trajectory(rig, seed=SEED, duration_s=DUR)
    both[rig.name] = NS.render(rig, traj, seed=SEED, n_mics=1)

for name, frame in both.items():
    m = dict(frame["meta"].items())
    print(f"{name:9s} rps {m['rps_min']:6.1f}-{m['rps_max']:6.1f} rev/s   rms {m['rms'][0]:.4f}")

NS.show(both, figsize=(14, 12))

## 4 · The knobs

One call, every knob. Edit and re-run.

* `seed` — moves both the trajectory draw and the render draw.
* `duration_s` — keep it ≤ 30 s; this is a laptop.
* `mean_shift` / `mean_scale` — move the rig's hover level (`mu * scale + shift`,
  rev/s); the ESC clamp and the warm-up idle level travel with it.
* `full_flight` — ground-to-ground envelope instead of the airborne process only.
* `traj_rig` — which drone to fly. `None` is the noise rig's own fitted rig; any
  name from `NS.traj_rig_names()` works; **`"posterior"`** draws a *fresh* drone
  from the global rig hyperprior (its own `mu`, trim, dynamics, level-offset
  model, idle level and ESC clamp), exactly as `rigs: {posterior: 1}` does in
  the training stream.
* `n_mics` — 1 to listen, up to 8 for the array.
* `comb_offset_db` — dB added to every rotor's `profile_db` (of a deep copy).

A hyperprior drone hovers anywhere from ~40 to ~300 rev/s (5th–95th
percentile; median ~137), so it usually leaves the 51.6–90.0 rev/s span the
DREGON fit was identified on and the render extrapolates `profile.amp_exp` and
`floor.floor_exp`. That is **reported, never clamped** — the warning and
`describe_draw` below print both ranges.

In [ ]:
clip = NS.sample_clip(
    "dregon",
    seed=3,
    duration_s=10.0,
    mean_shift=0.0,
    mean_scale=1.0,
    full_flight=False,
    traj_rig="posterior",   # None -> the rig's own fitted trajectory
    n_mics=1,
    comb_offset_db=0.0,
)
NS.describe_draw(clip)
NS.show(clip)